<a id="pca-objective"></a>

$$
\text{主成分分析 (Principal Component Analysis, PCA): 找方差最大的投影方向, 用少数几个方向概括数据} \\ \quad \\
\text{数据 } X \in \mathbb{R}^{n \times p} \text{, 先中心化 (每列减去均值) 得到 } X_c \text{, 样本协方差矩阵 } S = \frac{1}{n - 1} X_c^T X_c \\
\text{把每个样本投影到单位向量 w 上: } z_i = x_i^T w \text{ (投影长度), 所有样本 } z = X_c w \\
Var(z) = \frac{1}{n - 1} z^T z = \frac{1}{n - 1} w^T X_c^T X_c w = w^T S w \\ \quad \\
\text{第一主成分: } \max_w \ w^T S w \quad \text{s.t.} \quad w^T w = 1 \quad (\text{不限制长度的话, w 越长方差越大}) \\
\text{拉格朗日函数: } L = w^T S w - \lambda (w^T w - 1) \\
\nabla_w L = 2 S w - 2 \lambda w = 0 \implies S w = \lambda w \quad (\text{w 必须是 S 的特征向量}) \\
\text{此时方差 } w^T S w = w^T \lambda w = \lambda \implies \text{选最大的特征值 } \lambda_1 \text{ 对应的特征向量} \\ \quad \\
\text{第二主成分: 再加一个约束, 和第一主成分正交 } w^T w_1 = 0 \\
L = w^T S w - \lambda (w^T w - 1) - \mu \, w^T w_1 \implies 2 S w - 2 \lambda w - \mu w_1 = 0 \\
\text{左乘 } w_1^T \text{: } 2 w_1^T S w - 2\lambda \underbrace{w_1^T w}_{0} - \mu \underbrace{w_1^T w_1}_{1} = 0 \text{, 而 } w_1^T S w = (S w_1)^T w = \lambda_1 w_1^T w = 0 \\
\implies \mu = 0 \implies S w = \lambda w \text{, 选第二大的特征值} \\
\text{依此类推: 第 k 个主成分 = S 的第 k 大特征值的特征向量 (S 对称, 特征向量两两正交)}
$$

数学基础: [协方差矩阵与 Var(Xw) = w^T S w](../math_basics/statistics.ipynb#covariance-matrix) · [拉格朗日乘子法](../math_basics/calculus.ipynb#lagrange) · [对称矩阵的特征分解](../math_basics/linear_algebra.ipynb#symmetric-matrix)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

# 二维例子: 扫描所有方向, 方差最大的方向就是最大特征值的特征向量
rng = np.random.default_rng(0)
X2 = rng.multivariate_normal([2, 1], [[3.0, 1.5], [1.5, 1.2]], 500)
X2c = X2 - X2.mean(axis=0)
S2 = X2c.T @ X2c / (len(X2) - 1)

angles = np.linspace(0, np.pi, 1801)
dirs = np.column_stack([np.cos(angles), np.sin(angles)])
var_dir = np.einsum('ij,jk,ik->i', dirs, S2, dirs)        # 每个方向的 w^T S w
eigval2, eigvec2 = np.linalg.eigh(S2)
print(f'扫描得到的最大方差 = {var_dir.max():.6f}, 最大特征值 = {eigval2[-1]:.6f}')
print(f'最大方差方向 = {dirs[np.argmax(var_dir)]}, 特征向量 = {eigvec2[:, -1] * np.sign(eigvec2[0, -1])}')
print(f'投影后的样本方差 = {np.var(X2c @ eigvec2[:, -1], ddof=1):.6f}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].scatter(X2c[:, 0], X2c[:, 1], s=4, alpha=0.4)
for k, c in zip([1, 0], ['r', 'g']):
    v = eigvec2[:, k] * np.sqrt(eigval2[k]) * 2
    axs[0].plot([0, v[0]], [0, v[1]], c, linewidth=2, label=f'PC{2 - k}, lambda={eigval2[k]:.2f}')
axs[0].set_aspect('equal')
axs[1].plot(np.degrees(angles), var_dir, label='w^T S w')
axs[1].axhline(eigval2[-1], color='r', linestyle='--', label='lambda_1')
axs[1].axhline(eigval2[0], color='g', linestyle='--', label='lambda_2')
axs[1].set_xlabel('angle of w (degree)')
plot_nicely()
plt.show()

<a id="pca-svd"></a>

$$
\text{PCA 和 SVD 的关系 (实际计算用 SVD, 不需要先算 } X_c^T X_c \text{, 数值更稳定)} \\
X_c = U \Sigma V^T \implies X_c^T X_c = V \Sigma U^T U \Sigma V^T = V \Sigma^2 V^T \\
S = \frac{1}{n - 1} X_c^T X_c = V \frac{\Sigma^2}{n - 1} V^T \quad (\text{这就是 S 的特征分解}) \\ \quad \\
\text{结论:} \\
1. \ \text{主成分方向 = V 的列 (右奇异向量)} \\
2. \ \text{特征值 } \lambda_i = \frac{s_i^2}{n - 1} \\
3. \ \text{主成分得分 (scores) } Z = X_c V = U \Sigma V^T V = U \Sigma \\ \quad \\
\text{解释方差比 (Explained Variance Ratio): 总方差 = 各变量方差之和 = } \text{tr}(S) = \sum_i \lambda_i \text{ (迹 = 特征值之和)} \\
\text{第 i 个主成分解释的比例 } \frac{\lambda_i}{\sum_j \lambda_j} \text{, 前 k 个累计 } \frac{\sum_{i \leq k} \lambda_i}{\sum_j \lambda_j} \\ \quad \\
\text{符号不唯一: 若 v 是特征向量, -v 也是. 不同软件的符号可能相反, 比较时要先对齐符号}
$$

数学基础: [奇异值分解](../math_basics/linear_algebra.ipynb#svd) · [迹 = 特征值之和](../math_basics/linear_algebra.ipynb#trace)

In [ ]:
iris = load_iris()
X = iris.data
n = len(X)
Xc = X - X.mean(axis=0)

# 方法 1: 协方差矩阵特征分解 (eigh 返回升序, 翻转成降序)
S = Xc.T @ Xc / (n - 1)
eigval, eigvec = np.linalg.eigh(S)
eigval, eigvec = eigval[::-1], eigvec[:, ::-1]

# 方法 2: SVD
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
V = Vt.T

def align_sign(A, B):
    # 让 A 的每一列和 B 的对应列同号
    return A * np.sign(np.sum(A * B, axis=0))

print(f'特征值 = {eigval}')
print(f's^2 / (n - 1) = {s**2 / (n - 1)}, 一致? {np.allclose(eigval, s**2 / (n - 1))}')
print(f'特征向量 == V (对齐符号后)? {np.allclose(align_sign(eigvec, V), V)}')
print(f'X_c V == U Sigma ? {np.allclose(Xc @ V, U * s)}')
print(f'tr(S) = {np.trace(S):.6f}, 特征值之和 = {eigval.sum():.6f}')

ratio = eigval / eigval.sum()
print(f'解释方差比 = {np.round(ratio, 4)}, 累计 = {np.round(np.cumsum(ratio), 4)}')

sk = PCA().fit(X)
print(f'sklearn explained_variance_ == 特征值? {np.allclose(sk.explained_variance_, eigval)}')
print(f'sklearn explained_variance_ratio_ == 手算? {np.allclose(sk.explained_variance_ratio_, ratio)}')
print(f'sklearn components_ == V^T (对齐符号)? {np.allclose(align_sign(sk.components_.T, V), V)}')
print(f'sklearn transform == X_c V (对齐符号)? {np.allclose(align_sign(sk.transform(X), Xc @ V), Xc @ V)}')

<a id="pca-reconstruction"></a>

$$
\text{降维与重构 (Reconstruction)} \\
\text{只保留前 k 个主成分 } V_k \in \mathbb{R}^{p \times k} \text{: 降维 } Z_k = X_c V_k \text{, 重构 } \hat{X}_c = Z_k V_k^T = X_c V_k V_k^T \\
(V_k V_k^T \text{ 是投影到前 k 个主成分张成的子空间的投影矩阵)} \\ \quad \\
\text{重构误差: V 是正交矩阵, } V V^T = I \text{, 把 V 拆成保留的 } V_k \text{ 和丢掉的 } V_r \\
X_c = X_c V V^T = X_c V_k V_k^T + X_c V_r V_r^T \implies X_c - \hat{X}_c = X_c V_r V_r^T \\
\| X_c - \hat{X}_c \|_F^2 = \text{tr}(V_r V_r^T X_c^T X_c V_r V_r^T) = \text{tr}(V_r^T X_c^T X_c V_r \underbrace{V_r^T V_r}_{I}) \quad (\text{迹的循环性质}) \\
= \sum_{i > k} v_i^T X_c^T X_c v_i = \sum_{i > k} (n - 1) \lambda_i \\
\text{重构误差 = (n - 1) × 丢掉的特征值之和: 丢掉方差最小的方向, 损失的信息最少} \\
\text{(Eckart-Young 定理: 这也是所有秩为 k 的矩阵里离 } X_c \text{ 最近的, 即截断 SVD 的低秩近似)}
$$

数学基础: [投影矩阵](../math_basics/linear_algebra.ipynb#projection) · [正交矩阵](../math_basics/linear_algebra.ipynb#orthogonal) · [低秩近似](../math_basics/linear_algebra.ipynb#svd)

In [ ]:
for k in range(1, 5):
    Vk = V[:, :k]
    X_hat = Xc @ Vk @ Vk.T
    err = np.sum((Xc - X_hat)**2)
    print(f'k = {k}: 重构误差 = {err:.6f}, (n - 1) * 丢掉的特征值之和 = {(n - 1) * eigval[k:].sum():.6f}')

# sklearn 的重构 (inverse_transform 会把均值加回去)
sk2 = PCA(n_components=2).fit(X)
X_hat_sk = sk2.inverse_transform(sk2.transform(X))
X_hat_manual = Xc @ V[:, :2] @ V[:, :2].T + X.mean(axis=0)
print(f'k = 2 重构 手算 == sklearn? {np.allclose(X_hat_manual, X_hat_sk)}')

Z = Xc @ V[:, :2]
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for c, name in enumerate(iris.target_names):
    axs[0].scatter(Z[iris.target == c, 0], Z[iris.target == c, 1], s=8, label=name)
axs[0].set_xlabel(f'PC1 ({ratio[0]:.1%})')
axs[0].set_ylabel(f'PC2 ({ratio[1]:.1%})')
axs[1].bar(np.arange(1, 5), ratio, alpha=0.6, label='explained variance ratio')
axs[1].plot(np.arange(1, 5), np.cumsum(ratio), 'ro-', label='cumulative')
axs[1].set_xlabel('component')
plot_nicely()
plt.show()

$$
\text{注意: 量纲不同的变量要先标准化} \\
\text{PCA 最大化的是方差, 如果某个变量的单位让它的数值特别大 (例如以毫米记录), 第一主成分几乎就是这个变量} \\
\text{标准化后再做 PCA, 等价于对相关系数矩阵做特征分解}
$$

数学基础: [标准化](../math_basics/statistics.ipynb#standardization) · [协方差矩阵与相关系数矩阵](../math_basics/statistics.ipynb#covariance-matrix)

In [ ]:
X_mm = X.copy()
X_mm[:, 0] *= 1000                  # 第一个变量换成很小的单位, 数值放大 1000 倍
v1_raw = PCA(1).fit(X_mm).components_[0]
Z_std = (X_mm - X_mm.mean(axis=0)) / X_mm.std(axis=0, ddof=1)
v1_std = PCA(1).fit(Z_std).components_[0]
eig_corr = np.linalg.eigh(np.corrcoef(X_mm, rowvar=False))
print(f'不标准化: 第一主成分 = {np.round(np.abs(v1_raw), 4)} (几乎只有第一个变量)')
print(f'标准化后: 第一主成分 = {np.round(np.abs(v1_std), 4)}')
print(f'== 相关系数矩阵的最大特征向量? {np.allclose(np.abs(v1_std), np.abs(eig_corr[1][:, -1]))}')